# Regression strategies, revisited

**Regression strategies · version 2 · 11 November 2026.** Version 1 is `regression_models_comparison.ipynb` (git tag `v1`), unchanged. History: `VERSIONS.md`.

The original project (`regression_models_comparison.ipynb`, tagged `v1`) asked whether simple models can predict tomorrow's direction across the S&P 500. It is left untouched. This notebook keeps the same question, features and model families, and fixes the weaknesses of the test **one at a time**, so the effect of each fix can be read off:

| Step | Change | Weakness it removes |
|---|---|---|
| 0 | none: the original method, re-run on a frozen data snapshot | |
| 1 | a company counts only from the date it joined the index | look-ahead in the universe |
| 2 | 5 basis points charged on every dollar traded | free trading |
| 3 | rank each stock against its own sector; trade the top fifth against the bottom fifth | results driven by time in the market |
| 4 | predict the next 21 days, rebalance monthly | a daily horizon that costs cannot survive |

**Two limits, stated up front.**
- The data ends on 31 December 2021. Later years are held back for a separate project's final test, so this notebook cannot read them. The test period here is September 2018 to December 2021 (the original ran to 2026).
- The snapshot starts in 2006 (the original started in 2000), and companies that *left* the index are still missing: free data does not have them.

Because that test period was already seen in the original project, this is an improved **method**, not a fresh out-of-sample result.

In [1]:
# Imports. cross_section holds the new tools (plain functions on dates x tickers frames);
# func_lib is imported for the shared dark chart theme and the table figure.
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

for path in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..'))):
    if os.path.exists(os.path.join(path, 'func_lib.py')) and path not in sys.path:
        sys.path.append(path)
import cross_section as cs
import func_lib

warnings.filterwarnings('ignore', category=FutureWarning)

In [2]:
# Configuration. The momentum windows and model families are the original's.
MOMENTUM_WINDOWS = [1, 5, 15, 20]
FEATURES = [f'{m}_d_returns' for m in MOMENTUM_WINDOWS]
TRAIN = ('2007-02-01', '2018-08-31')
TEST = ('2018-09-01', '2021-12-31')
COST_BPS = 5.0             # charged on every dollar bought or sold
QUANTILE = 0.2             # long the top fifth of each sector, short the bottom fifth
RANDOM_STATE = 42
LASSO_ALPHAS = np.logspace(-7, -2, 11)
MODELS = ['Linear', 'Lasso', 'Logistic', 'Tree']

## 1. Data: the same prices, two definitions of the universe

In [3]:
# Frozen snapshot of adjusted closes. `everyone` is the original practice (each current member for
# its whole history); `pit` admits a company only from the date it joined the index.
everyone = cs.load_sp500_panel(point_in_time=False)
pit = cs.load_sp500_panel(point_in_time=True)
close, sectors = pit.close, pit.sectors
fwd1 = cs.forward_return(close, 1)
momentum = cs.momentum_features(close, MOMENTUM_WINDOWS)

count = pd.DataFrame({'original universe': everyone.member.sum(axis=1), 'joined the index by that date': pit.member.sum(axis=1)})
print(f'{close.shape[1]} tickers, {close.index[0]:%Y-%m-%d} to {close.index[-1]:%Y-%m-%d}')
print(count.resample('YE').mean().round(0).astype(int).iloc[::3].to_string())
fig = go.Figure([go.Scatter(x=count.index, y=count[c], name=c) for c in count])
fig.update_layout(title='Stocks in the universe each day', yaxis_title='stocks', height=380)
fig.show()

503 tickers, 2006-01-03 to 2021-12-31
            original universe  joined the index by that date
Date                                                        
2006-12-31                394                            222
2009-12-31                414                            259
2012-12-31                438                            288
2015-12-31                460                            321
2018-12-31                475                            377
2021-12-31                491                            420


In 2007 the original universe holds about 400 stocks, but only about 270 of them were in the index then. The other 130 are companies we know *today* went on to join it, which mostly means they did well. A model trained on them has been told something about the future.

## 2. One fitting routine for every step

In [4]:
def fit_models(train_x, train_y, train_up):
    """The four model families. Regressions forecast the target; classifiers the chance it is positive.
    Lasso's penalty is chosen on the last fifth of the training dates (never on test data)."""
    dates = train_x.index.get_level_values('Date')
    cut = dates.unique().sort_values()[int(0.8 * dates.nunique())]
    a, b = dates < cut, dates >= cut
    mse = {al: float(((Lasso(alpha=al, max_iter=5000).fit(train_x[a], train_y[a]).predict(train_x[b]) - train_y[b]) ** 2).mean())
           for al in LASSO_ALPHAS}
    best = min(mse, key=mse.get)
    return {'Linear': LinearRegression().fit(train_x, train_y),
            'Lasso': Lasso(alpha=best, max_iter=5000).fit(train_x, train_y),
            'Logistic': LogisticRegression(max_iter=200).fit(train_x, train_up),
            'Tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=2000, random_state=RANDOM_STATE).fit(train_x, train_up)}, best


def score_models(models, test_x, like):
    """Each model's score on the test rows as a dates x tickers frame. Regressions: the forecast.
    Classifiers: P(up) - 0.5, so that zero is the long-or-cash threshold for every model."""
    out = {}
    for name, m in models.items():
        s = m.predict(test_x) if name in ('Linear', 'Lasso') else m.predict_proba(test_x)[:, 1] - 0.5
        out[name] = cs.unstack_scores(s, test_x.index, like)
    return out


def run(member, features, target, h=1, every=1):
    """Fit on TRAIN and score on TEST for one universe / feature set / target.
    The last h training days are dropped so no training target reaches into the test period."""
    frames = {**features, 'target': target}
    train_end = close.index[close.index.searchsorted(pd.Timestamp(TRAIN[1]), side='right') - 1 - h]
    tr = cs.stack(frames, member, (pd.Timestamp(TRAIN[0]), train_end), every=every)
    te = cs.stack(features, member, (pd.Timestamp(TEST[0]), pd.Timestamp(TEST[1])))
    scaler = StandardScaler().fit(tr[FEATURES])
    x_tr = pd.DataFrame(scaler.transform(tr[FEATURES]), index=tr.index, columns=FEATURES)
    x_te = pd.DataFrame(scaler.transform(te[FEATURES]), index=te.index, columns=FEATURES)
    models, alpha = fit_models(x_tr, tr['target'], (tr['target'] > 0).astype(int))
    return {'scores': score_models(models, x_te, close.loc[TEST[0]:TEST[1]]), 'models': models, 'rows': len(tr), 'alpha': alpha}

## 3. Steps 0 to 2: the original rule, made honest

The original rule: hold a stock when its forecast is positive, otherwise hold cash; average over all stocks. The benchmark holds every stock every day.

In [5]:
test_idx = close.loc[TEST[0]:TEST[1]].index[:-1]            # the last day has no next-day return
fwd_test = fwd1.reindex(test_idx)


def long_flat_table(res, member, cost_bps):
    bm = cs.benchmark_returns(fwd_test, member.reindex(test_idx))
    rows = {'Benchmark': {**cs.perf(bm), 'time_long': 1.0, 'turnover_per_year': 0.0, 't_vs_benchmark': np.nan}}
    for name in MODELS:
        lf = cs.long_flat_returns(res['scores'][name].reindex(test_idx), fwd_test, member.reindex(test_idx), cost_bps=cost_bps)
        rows[name] = {**cs.perf(lf['ret']), 'time_long': float(lf['time_long'].mean()),
                      'turnover_per_year': float(lf['turnover'].mean() * 252), 't_vs_benchmark': cs.active_t(lf['ret'], bm)}
    return pd.DataFrame(rows).T


step0 = run(everyone.member, momentum, fwd1)                # original universe
step1 = run(pit.member, momentum, fwd1)                     # joined-by-that-date universe
tables = {'0. Original method': long_flat_table(step0, everyone.member, 0.0),
          '1. + index membership by date': long_flat_table(step1, pit.member, 0.0),
          '2. + 5 bp trading costs': long_flat_table(step1, pit.member, COST_BPS)}
cols = ['growth_of_1', 'ann_return', 'sharpe', 'max_drawdown', 'time_long', 'turnover_per_year', 't_vs_benchmark']
for name, t in tables.items():
    print(f"\n{name}  (training rows: {(step0 if name.startswith('0') else step1)['rows']:,})")
    print(t[cols].round(3).to_string())


0. Original method  (training rows: 1,283,430)
           growth_of_1  ann_return  sharpe  max_drawdown  time_long  turnover_per_year  t_vs_benchmark
Benchmark        1.932       0.227   0.949         0.383      1.000              0.000             NaN
Linear           1.630       0.168   0.826         0.342      0.773             45.725          -1.472
Lasso            1.942       0.228   0.956         0.383      1.000              0.386           1.258
Logistic         1.862       0.209   0.993         0.326      0.879             35.972          -0.534
Tree             1.948       0.216   1.221         0.214      0.682             89.746          -0.173

1. + index membership by date  (training rows: 863,727)
           growth_of_1  ann_return  sharpe  max_drawdown  time_long  turnover_per_year  t_vs_benchmark
Benchmark        1.802       0.206   0.862         0.387      1.000              0.000             NaN
Linear           1.588       0.159   0.806         0.326      0.716    

In [6]:
# Growth of $1 for the best original model at each step, against the benchmark of the same universe
best = tables['0. Original method'].drop('Benchmark')['growth_of_1'].idxmax()
fig = go.Figure()
for label, res, member, cost, dash in (('0. original method', step0, everyone.member, 0.0, 'solid'),
                                       ('1. + membership by date', step1, pit.member, 0.0, 'dash'),
                                       ('2. + costs', step1, pit.member, COST_BPS, 'dot')):
    lf = cs.long_flat_returns(res['scores'][best].reindex(test_idx), fwd_test, member.reindex(test_idx), cost_bps=cost)
    fig.add_scatter(x=lf.index, y=(1 + lf['ret']).cumprod(), name=f'{best}: {label}', line=dict(dash=dash))
bm = cs.benchmark_returns(fwd_test, pit.member.reindex(test_idx))
fig.add_scatter(x=bm.index, y=(1 + bm).cumprod(), name='Benchmark (membership by date)', line=dict(color='#9aa1ab', width=1.2))
fig.update_layout(title=f'Growth of $1, {TEST[0][:7]} to {TEST[1][:7]}: {best} at each step', yaxis_title='growth of $1', height=440)
fig.show()

## 4. Step 3: rank stocks against their own sector

The long-or-cash rule mostly measures how much of the time a model is invested. To test *prediction*, remove the market and the sector: every feature and the target become the stock's rank inside its sector that day, and the strategy buys the top fifth of each sector and sells the bottom fifth. It makes money only if the model orders stocks correctly.

The **information coefficient (IC)** is the daily correlation between the model's ranking and the ranking of what happened next. Zero means no skill; 0.02 is a useful signal *if* it can be traded cheaply.

In [7]:
def ranked(frames, member):
    return {k: cs.sector_rank(v, sectors, member) for k, v in frames.items()}


def long_short_table(res, target_fwd, h, every):
    rows = {}
    for name in MODELS:
        sc = res['scores'][name].reindex(test_idx)
        ic = cs.ic_summary(cs.daily_rank_ic(sc, target_fwd.reindex(test_idx), sectors, pit.member.reindex(test_idx)), h)
        gross = cs.long_short_returns(sc, fwd_test, sectors, pit.member.reindex(test_idx), QUANTILE, every, 0.0)
        net = cs.long_short_returns(sc, fwd_test, sectors, pit.member.reindex(test_idx), QUANTILE, every, COST_BPS)
        rows[name] = {'IC': ic['IC'], 't': ic['t'], 'sharpe_before_costs': cs.perf(gross['ret'])['sharpe'],
                      'sharpe_after_costs': cs.perf(net['ret'])['sharpe'], 'ann_return_after_costs': cs.perf(net['ret'])['ann_return'],
                      'turnover_per_year': float(net['turnover'].mean() * 252)}
    return pd.DataFrame(rows).T


rank_feats = ranked(momentum, pit.member)
step3 = run(pit.member, rank_feats, cs.sector_rank(fwd1, sectors, pit.member))
tables['3. Sector-relative, long-short, daily'] = long_short_table(step3, fwd1, 1, 1)
print(tables['3. Sector-relative, long-short, daily'].round(3).to_string())
print('\nLinear model weights on the ranked features:', dict(zip(FEATURES, step3['models']['Linear'].coef_.round(4))))

             IC      t  sharpe_before_costs  sharpe_after_costs  ann_return_after_costs  turnover_per_year
Linear    0.013  2.126                0.759              -1.223                  -0.201            651.297
Lasso     0.014  2.230                0.828              -1.166                  -0.192            656.299
Logistic  0.014  2.184                0.757              -1.227                  -0.202            654.497
Tree      0.011  1.976                0.776              -1.542                  -0.237            711.768

Linear model weights on the ranked features: {'1_d_returns': np.float64(-0.0038), '5_d_returns': np.float64(-0.0014), '15_d_returns': np.float64(-0.0024), '20_d_returns': np.float64(0.0004)}


## 5. Step 4: a 21-day horizon, rebalanced monthly

Daily trading turns the book over hundreds of times a year. Predicting the next 21 days and rebalancing monthly cuts trading by roughly twentyfold. Training uses every fifth day (21-day targets on neighbouring days overlap almost entirely), and the t-statistic counts one independent observation per 21 days.

In [8]:
H = 21
fwd21 = cs.forward_return(close, H)
step4 = run(pit.member, rank_feats, cs.sector_rank(fwd21, sectors, pit.member), h=H, every=5)
tables['4. 21-day horizon, monthly'] = long_short_table(step4, fwd21, H, H)
print(tables['4. 21-day horizon, monthly'].round(3).to_string())
print('\nLinear model weights on the ranked features:', dict(zip(FEATURES, step4['models']['Linear'].coef_.round(4))))

             IC      t  sharpe_before_costs  sharpe_after_costs  ann_return_after_costs  turnover_per_year
Linear    0.021  0.799                0.594               0.477                   0.079             38.664
Lasso     0.021  0.799                0.594               0.477                   0.079             38.664
Logistic  0.021  0.827                0.468               0.342                   0.053             38.792
Tree     -0.000 -0.026                0.157              -0.168                  -0.010             40.425

Linear model weights on the ranked features: {'1_d_returns': np.float64(-0.0007), '5_d_returns': np.float64(-0.0034), '15_d_returns': np.float64(-0.0005), '20_d_returns': np.float64(0.0)}


## 6. The ladder in one table

In [9]:
# One line per step for the model that did best in the original method, and the best model at each step
def pick(t, name, col):
    return float(t.loc[name, col])


ladder = []
for step in ('0. Original method', '1. + index membership by date', '2. + 5 bp trading costs'):
    t = tables[step]
    top = t.drop('Benchmark')['growth_of_1'].idxmax()
    ladder.append({'step': step, 'best model': top, 'growth of $1': pick(t, top, 'growth_of_1'), 'benchmark': pick(t, 'Benchmark', 'growth_of_1'),
                   'Sharpe': pick(t, top, 'sharpe'), 'benchmark Sharpe': pick(t, 'Benchmark', 'sharpe'), 't vs benchmark': pick(t, top, 't_vs_benchmark'),
                   'time long': pick(t, top, 'time_long')})
for step in ('3. Sector-relative, long-short, daily', '4. 21-day horizon, monthly'):
    t = tables[step]
    top = t['sharpe_after_costs'].idxmax()
    ladder.append({'step': step, 'best model': top, 'IC': pick(t, top, 'IC'), 't of IC': pick(t, top, 't'),
                   'Sharpe before costs': pick(t, top, 'sharpe_before_costs'), 'Sharpe after costs': pick(t, top, 'sharpe_after_costs'),
                   'turnover per year': pick(t, top, 'turnover_per_year')})
ladder = pd.DataFrame(ladder).set_index('step')
print(ladder.round(3).to_string())

# Saved for the website's 'Revisited' block (numbers on the site are read from this file, not typed)
summary = {'test_period': TEST, 'train_period': TRAIN, 'cost_bps': COST_BPS,
           'universe_2007': {'original': int(everyone.member.loc['2007'].sum(axis=1).mean()), 'by_date': int(pit.member.loc['2007'].sum(axis=1).mean())},
           'tables': {k: json.loads(v.to_json(orient='index')) for k, v in tables.items()},
           'ladder': json.loads(ladder.reset_index().to_json(orient='records'))}
with open(os.path.join('data_cache', 'regression_revisited_summary.json'), 'w') as f:
    json.dump(summary, f, indent=1)

                                      best model  growth of $1  benchmark  Sharpe  benchmark Sharpe  t vs benchmark  time long     IC  t of IC  Sharpe before costs  Sharpe after costs  turnover per year
step                                                                                                                                                                                                      
0. Original method                          Tree         1.948      1.932   1.221             0.949          -0.173      0.682    NaN      NaN                  NaN                 NaN                NaN
1. + index membership by date              Lasso         1.854      1.802   0.905             0.862           1.146      0.997    NaN      NaN                  NaN                 NaN                NaN
2. + 5 bp trading costs                    Lasso         1.850      1.802   0.903             0.862           1.063      0.997    NaN      NaN                  NaN                 NaN     

## 7. Reading the results

How to read the tables; the findings themselves are printed by the next cell from the computed numbers.

- **Steps 0 to 2** use the original long-or-cash rule. Compare each model with the benchmark *of the same step*. `t_vs_benchmark` below 2 in absolute value means the difference is within noise. `time_long` shows how much of a result is simply being invested.
- **Step 1** changes the universe. Any drop from step 0 is return that came from knowing which companies would later join the index.
- **Step 2** charges costs: `turnover_per_year` times 5 basis points is the annual bill.
- **Step 3** is the first real test of prediction: `IC` and its `t` say whether the ranking has skill; the two Sharpe columns say whether it survives trading.
- **Step 4** asks the same at a horizon where costs are small.

Still open: companies that left the index are missing, the test period was seen before, and four past returns are the only inputs.

In [10]:
# The findings, written from the numbers above (nothing typed by hand)
t0, t1, t2 = (tables[k] for k in ('0. Original method', '1. + index membership by date', '2. + 5 bp trading costs'))
t3, t4 = tables['3. Sector-relative, long-short, daily'], tables['4. 21-day horizon, monthly']
m0 = t0.drop('Benchmark')
years = len(test_idx) / 252
print(f"Test period: {test_idx[0]:%b %Y} to {test_idx[-1]:%b %Y} ({years:.1f} years).\n")
print(f"1. No model's result differs from the benchmark by more than noise: the largest |t| against the benchmark "
      f"in steps 0 to 2 is {max(t.drop('Benchmark')['t_vs_benchmark'].abs().max() for t in (t0, t1, t2)):.1f} (2 is the usual bar).")
print(f"2. Universe look-ahead: the benchmark's growth of $1 falls from {t0.loc['Benchmark', 'growth_of_1']:.2f} to "
      f"{t1.loc['Benchmark', 'growth_of_1']:.2f} when companies count only from their index-join date "
      f"({100 * (t0.loc['Benchmark', 'ann_return'] - t1.loc['Benchmark', 'ann_return']):.1f} points a year).")
top0 = m0['sharpe'].idxmax()
print(f"3. Costs: the best Sharpe ratio in step 0 ({top0}, {m0.loc[top0, 'sharpe']:.2f} against the benchmark's "
      f"{t0.loc['Benchmark', 'sharpe']:.2f}) becomes {t2.loc[top0, 'sharpe']:.2f} against {t2.loc['Benchmark', 'sharpe']:.2f} "
      f"after steps 1 and 2; it trades {t2.loc[top0, 'turnover_per_year']:.0f} times its capital a year.")
b3 = t3['IC'].idxmax()
print(f"4. Ranked against their sector, the models DO have a little daily skill: IC {t3.loc[b3, 'IC']:.3f} (t {t3.loc[b3, 't']:.1f}) for {b3}. "
      f"It earns a Sharpe of {t3.loc[b3, 'sharpe_before_costs']:.2f} before costs and {t3.loc[b3, 'sharpe_after_costs']:.2f} after, "
      f"because the book turns over {t3.loc[b3, 'turnover_per_year']:.0f} times a year.")
w = dict(zip(FEATURES, step3['models']['Linear'].coef_))
print(f"   The weight on the 1-day return is {w['1_d_returns']:.4f}: the skill is short-term REVERSAL, not momentum.")
b4 = t4['sharpe_after_costs'].idxmax()
se = 1 / np.sqrt(years)
print(f"5. At 21 days, rebalanced monthly: IC {t4.loc[b4, 'IC']:.3f} (t {t4.loc[b4, 't']:.1f}), Sharpe {t4.loc[b4, 'sharpe_after_costs']:.2f} after costs "
      f"at {t4.loc[b4, 'turnover_per_year']:.0f}x turnover. Over {years:.1f} years a Sharpe ratio has a standard error of about {se:.2f}, "
      f"so this is {'not ' if abs(t4.loc[b4, 'sharpe_after_costs']) < 2 * se else ''}distinguishable from zero. "
      f"Costs here are 5 bp on trades only; borrowing shares to short is not charged.")

Test period: Sep 2018 to Dec 2021 (3.3 years).

1. No model's result differs from the benchmark by more than noise: the largest |t| against the benchmark in steps 0 to 2 is 1.7 (2 is the usual bar).
2. Universe look-ahead: the benchmark's growth of $1 falls from 1.93 to 1.80 when companies count only from their index-join date (2.1 points a year).
3. Costs: the best Sharpe ratio in step 0 (Tree, 1.22 against the benchmark's 0.95) becomes 0.85 against 0.86 after steps 1 and 2; it trades 83 times its capital a year.
4. Ranked against their sector, the models DO have a little daily skill: IC 0.014 (t 2.2) for Lasso. It earns a Sharpe of 0.83 before costs and -1.17 after, because the book turns over 656 times a year.
   The weight on the 1-day return is -0.0038: the skill is short-term REVERSAL, not momentum.
5. At 21 days, rebalanced monthly: IC 0.021 (t 0.8), Sharpe 0.48 after costs at 39x turnover. Over 3.3 years a Sharpe ratio has a standard error of about 0.55, so this is not distingu